# Session 6 - Lab: the shopping assistant, and how you know it works

**DAT09-06 Representation Learning · Monday 23 November 2026 · 1 h 30**

Last week you built a search you can type at. Today it becomes an
**assistant**: a language model reads the customer's request, calls **your
search**, and writes its answer from the products that come back. Then you
measure it on 26 real requests, and you find out which part of the system
decides the score.

The language model is **Claude Haiku**, behind a small relay the teacher runs
for the class. You need the **class key** written on the board.

### The words of the lecture, and the names your AI will use

| In the lecture | In papers and in code |
|---|---|
| the **memory**: the search you built last week | the *retriever*, a *vector index* |
| **paste the products** into the request, then let the LLM answer | *retrieval-augmented generation* (RAG), *grounding* |
| the LLM **writes the search** itself | *tool use*, *function calling*; the search is a *tool* |
| the **filters**: colour, who it is for | *metadata filtering*, *structured* or *hybrid search* |
| the **test set** of 26 requests, and its **grade** | an *evaluation set*, an *end-to-end eval* |
| the **system text** that frames the assistant | the *system prompt* |

### How this notebook is marked

| Marker | What it means |
|---|---|
| `# Given - read it, run it, move on.` | nothing to write. Run the cell. |
| ✏️ **YOUR TURN** | a boxed banner tells you **how many lines** and **what they do**. That is the only place you write code. |
| ❓ **Checkpoint** | **three questions** about *your* output. Answer **in writing**, in the cell below. Most only ask you to read what you just printed. |
| ⭐ **Going further** | optional, harder. Only if you are ahead: the next checkpoint never needs them. |

There are **8 cells to write**: seven of them are at most five lines, the last
one is the open challenge. Everything else is given, including the loop that
lets the LLM call your search, the grader and every plot.

### Read this before you start

An AI will write all eight in two minutes. Let it - but **read what the
assistant answers**: the trap of the day is an answer that is polite, honest
about what it was given, and false. You meet it on purpose in checkpoint 2.

Every call costs the teacher a fraction of a cent, and the whole class shares
one key. Run the evaluations when a question asks for them, not in a loop.

### Six checkpoints

| | What you produce | ~ |
|---|---|---|
| **1** | The LLM alone: it speaks well and knows nothing of our shop | 10 min |
| **2** | Search, then answer - the score to beat, and the trap | 20 min |
| **3** | The LLM writes the search: a tool, with filters | 20 min |
| **4** | "We do not sell that" | 10 min |
| **5** | **The search is the weak link** - four memories, one table | 15 min |
| **6** | **The challenge** - your best assistant, on requests you have never seen | open |

**Most of you will not reach checkpoint 6. That is deliberate.**

Run the cells **in order**.

---

## 0 · Setup

Five given cells: imports, the connection to the LLM, the catalogue, last
week's search, and a helper to show products.

In [ ]:
# Given - read it, run it, move on.
import logging
import os
import re
import sys
import textwrap
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass, getuser
from pathlib import Path

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # a missing font is not news
os.environ.setdefault("HF_HUB_VERBOSITY", "error")                     # nor is an anonymous download
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.insert(0, str(Path.cwd().parent))   # make `common` importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

try:
    import anthropic
except ImportError:
    raise ImportError("The Anthropic SDK is missing. Run   pip install anthropic   in your "
                      "environment, then restart the kernel.") from None

from common.paths import PROCESSED, require
from common.plotting import PALETTE, use_course_style
from common.seeding import SEED, cpu_threads, set_seed

use_course_style()
set_seed(SEED)
cpu_threads()

print("torch", torch.__version__, "| anthropic", anthropic.__version__)

### 0.1 · The connection to the LLM

The teacher's relay holds the real API key. You talk to it with the official
Anthropic SDK, exactly as you would talk to the API itself, with two changes:
the address (`base_url`) and the key, which is the **class key** on the board.

The relay always answers with **Claude Haiku**, refuses streaming, and caps the
length of an answer. Your first name in `x-student` lets the teacher see in the
logs who is stuck.

We also ask Haiku to **think briefly** before it answers (`effort: low`): a
little faster and cheaper than its default, and enough for a shopping question.

In [ ]:
# Given - read it, run it, move on.
RELAY_URL = "https://gigpsqqohghwichjdvucbcmcua0pzora.lambda-url.eu-west-3.on.aws"
MODEL = "claude-haiku-5-5"

CLASS_KEY = os.environ.get("DAT09_CLASS_KEY") or getpass("Class key (written on the board): ")
STUDENT = os.environ.get("DAT09_STUDENT") or getuser()

client = anthropic.Anthropic(base_url=RELAY_URL, api_key=CLASS_KEY, max_retries=6,
                             default_headers={"x-student": STUDENT})

USAGE = {"calls": 0, "in": 0, "out": 0}
_usage_lock = threading.Lock()


def llm(**request):
    """One call to Claude Haiku through the class relay. Counts the tokens it used."""
    request.setdefault("max_tokens", 1024)
    request.setdefault("output_config", {"effort": "low"})
    reply = client.messages.create(model=MODEL, **request)
    with _usage_lock:
        USAGE["calls"] += 1
        USAGE["in"] += reply.usage.input_tokens
        USAGE["out"] += reply.usage.output_tokens
    return reply


def text_of(reply):
    """The text of a reply, without the blocks where the model thought."""
    return "".join(block.text for block in reply.content if block.type == "text")


t0 = time.time()
reply = llm(messages=[{"role": "user", "content": "Say hello to a class of data scientists, in six words."}])
print(text_of(reply))
print(f"{time.time() - t0:.1f} s   {reply.usage.input_tokens} tokens in, {reply.usage.output_tokens} out"
      f"   model: {reply.model}")

If this cell printed an `AuthenticationError`, the key is wrong: run it
again and type the key from the board. If it printed a `ConnectionError`,
check the wifi.

**Why `max_tokens` is 1,024 and not 50.** Haiku writes a short, hidden
reflection before its answer, and that reflection counts inside `max_tokens`.
With 50, the reply can be the reflection alone, with no text at all.

### 0.2 · The catalogue

The same 6,000 products as last week: twelve kinds, a photo and a title each,
and a few columns the shop filled in - colour, who it is for, usage.

In [ ]:
# Given - read it, run it, move on.
HINT = "git pull   (the product files ship with the repository)"

meta = pd.read_parquet(require(PROCESSED / "products_meta.parquet", hint=HINT))
kinds = sorted(meta["article_type"].unique())
ID_TO_ROW = {int(pid): row for row, pid in enumerate(meta["id"])}


def load_codes(name):
    """A table of frozen codes, one row per product, every row scaled to length 1."""
    table = np.load(require(PROCESSED / name, hint=HINT)).astype(np.float32)
    return F.normalize(torch.from_numpy(table), dim=1)


photo_codes = load_codes("products_image_emb.npy")     # (6000, 512)  the photo expert
title_codes = load_codes("products_text_emb.npy")      # (6000, 384)  the sentence expert
train = torch.tensor((meta["split"] == "train").to_numpy())

print("kinds we sell:", ", ".join(kinds))
print("colours:", ", ".join(meta["colour"].value_counts().index[:14]), "...")
meta[["id", "title", "article_type", "colour", "gender", "usage"]].head(4)

### 0.3 · Last week's search: the assistant's memory

You wrote all of this last week; here it is, given. Two small heads are trained
again with the matching game (about a second), and `search(text)` returns the
rows of the products whose **photo** is closest to the text in the shared space.

`keep` is new: a mask of the products the search is allowed to return. You use
it in checkpoint 3.

In [ ]:
# Given - read it, run it, move on.
class Heads(nn.Module):
    """One small layer per expert, both ending in the same 64 numbers: the shared space."""

    def __init__(self, d=64):
        super().__init__()
        self.photo = nn.Linear(512, d)
        self.text = nn.Linear(384, d)


def matching_loss(a, b, temperature=0.1):
    a, b = F.normalize(a, dim=1), F.normalize(b, dim=1)
    grid = a @ b.T / temperature
    target = torch.arange(len(a))
    return (F.cross_entropy(grid, target) + F.cross_entropy(grid.T, target)) / 2


def train_heads(seed=SEED, epochs=20):
    set_seed(seed)
    heads = Heads()
    opt = torch.optim.Adam(heads.parameters(), lr=1e-3)
    photos, titles = photo_codes[train], title_codes[train]
    for _ in range(epochs):
        perm = torch.randperm(len(photos))
        for i in range(0, len(photos), 256):
            idx = perm[i:i + 256]
            loss = matching_loss(heads.photo(photos[idx]), heads.text(titles[idx]))
            opt.zero_grad(); loss.backward(); opt.step()
    return heads.eval()


from sentence_transformers import SentenceTransformer
sentence_expert = SentenceTransformer("all-MiniLM-L6-v2")
_expert_lock = threading.Lock()


def sentence_code(sentence):
    """The sentence expert's 384 numbers for one sentence, scaled to length 1."""
    with _expert_lock:
        return F.normalize(torch.from_numpy(sentence_expert.encode([sentence]))[0], dim=0)


class PhotoMemory:
    """The 6,000 photos in the shared space. A request goes through the text head."""

    def __init__(self, heads, index_heads=None):
        self.heads = heads
        with torch.no_grad():       # the index: every photo through the photo head, once
            self.index = F.normalize((index_heads or heads).photo(photo_codes), dim=1)

    def similarities(self, text):
        with torch.no_grad():
            return self.index @ F.normalize(self.heads.text(sentence_code(text)), dim=0)


t0 = time.time()
heads = train_heads()
MEMORY = PhotoMemory(heads)
print(f"heads trained and 6,000 photos indexed in {time.time() - t0:.1f} s")


def search(text, k=8, keep=None, memory=None):
    """The rows of the k products closest to the text. `keep`: which products are allowed."""
    sim = (memory or MEMORY).similarities(text)
    if keep is not None:
        sim = torch.where(torch.as_tensor(keep), sim, torch.tensor(-9.0))
    return sim.topk(k).indices.tolist()


print("a red t-shirt ->", [meta["title"][r] for r in search("a red t-shirt", k=3)])

In [ ]:
# Given - read it, run it, move on.
THUMBS = PROCESSED / "products_images"


def show_products(rows, title=None):
    """Show the photos of these rows of the catalogue, with their titles underneath."""
    rows = [int(r) for r in rows]
    if not rows:
        print("(no product)"); return
    fig, axes = plt.subplots(1, len(rows), figsize=(len(rows) * 1.45, 2.5))
    for ax, r in zip(np.atleast_1d(axes), rows):
        ax.imshow(Image.open(THUMBS / f"{meta['id'][r]}.jpg"))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_xlabel(textwrap.fill(str(meta["title"][r]), 17), fontsize=7)
    if title:
        fig.suptitle(title, y=1.04, fontsize=11)
    plt.show()


def cited(answer):
    """The rows of the products an answer cites as [id], in order, each once."""
    ids = [int(m) for m in re.findall(r"\[(\d+)\]", answer)]
    return list(dict.fromkeys(ID_TO_ROW[i] for i in ids if i in ID_TO_ROW))


show_products(search("green flip flops", k=6), '"green flip flops"')

---

# Checkpoint 1 - The LLM alone

**~10 minutes.** One cell to write, three lines.

### 1.1 ✏️ · Ask the LLM

A request to the LLM has two texts: the **system text**, which says who it is
and what it must do, and the **customer's message**. Write `ask`: one call to
`llm(...)` with both, and return the text of the reply.

`llm` takes the same arguments as `client.messages.create`; the model, the
length and the thinking effort are already set.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     reply = llm(system=system, messages=[{"role": "user", "content": message}])
#     return text_of(reply)
# ──────────────────────────────────────────────────────────────────────
def ask(system, message):
    """One question to the LLM: a system text and one message. Returns the text of the answer."""
    ...          # <- replace this line


print(ask("You answer with one word.", "What colour is a ripe banana?"))

### 1.2 ▶ · Three customers, an LLM, and no catalogue

The system text says it works for our shop and must name three products.

In [ ]:
# Given - read it, run it, move on.
SHOP_ALONE = ("You are the shopping assistant of our online fashion shop. Always recommend "
              "exactly 3 products from our shop, by their exact product name.")

for request in ("I'm looking for a red t-shirt for men.",
                "A brown leather wallet for my dad.",
                "Une montre pour femme."):
    answer = ask(SHOP_ALONE, request)
    named = [t for t in meta["title"] if t.lower() in answer.lower()]
    print(f"### {request}\n{answer}\n-> products of our catalogue named: {len(named)}\n")

### ❓ Checkpoint 1

1. How many products of our catalogue did the three answers name? What did the
   LLM do instead?
2. The LLM writes fluent French and knows what a kurta is. Where does that
   knowledge come from, and why does it not contain our 6,000 products?
3. What, exactly, must we hand it so that it can answer the first request?

*Your answers:*

1.
2.
3.

---

# Checkpoint 2 - Search, then answer

**~20 minutes.** Two cells to write, five lines in all.

The simplest assistant there is: search the catalogue with the customer's
words, **paste** the products that come back into the message, and ask the LLM
to answer from them only.

### 2.1 ✏️ · A product, as one line of text

The LLM reads text. Turn rows of the catalogue into lines like

`[1541] Puma Men's Ballistic Spike White Green Shoe | Sports Shoes | colour: White | for: Men | Sports`

The number in brackets is the product's `id`: the assistant cites it, and
that is how we check its answer.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     one f-string per row r, with meta['id'][r], meta['title'][r], meta['article_type'][r],
#     meta['colour'][r], meta['gender'][r], meta['usage'][r] - joined with "\n"
# ──────────────────────────────────────────────────────────────────────
def as_lines(rows):
    """One line of text per product: [id] title | kind | colour | for | usage."""
    ...          # <- replace this line


print(as_lines(search("a red t-shirt", k=3)))

### 2.2 ✏️ · The first assistant

Search with the customer's own words, put the products and the request in one
message, and ask. Return the answer **and** the rows you pasted, so that we can
look at what the LLM was given.

In [ ]:
# Given - read it, run it, move on.
SYSTEM_PASTE = ("You are the shopping assistant of an online fashion shop. Answer the customer "
                "from the products below only. Recommend at most 3, and cite each one you "
                "recommend as [id]. If none fits, say so.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     rows = search(request, k=k)
#     message = f"Products:\n{as_lines(rows)}\n\nCustomer: {request}"
# ──────────────────────────────────────────────────────────────────────
def assistant_v1(request, k=8):
    """Search with the request, paste the k products, let the LLM answer. Returns (answer, rows)."""
    ...          # <- replace this line
    return ask(SYSTEM_PASTE, message), rows


answer, rows = assistant_v1("I'm looking for a red t-shirt for men.")
show_products(rows, "what the search found - and what the LLM was given")
print(answer)
show_products(cited(answer), "what the assistant recommends")

### 2.3 ▶ · The test set: 26 requests, and a grade

A demo on one request proves nothing. Here are **26 requests**, written once
and never changed: 18 for products we sell (12 in English, 6 in French) and 8
for products we do not sell.

The grade is mechanical, so that two runs can be compared:

- **we sell it**: the answer is right when it cites at least one product and
  **every** product it cites is of the right kind, colour and person;
- **we do not sell it**: the answer is right when it cites **no** product.

`evaluate` sends the 26 requests four at a time and prints one line each. It
takes about 20 seconds.

In [ ]:
# Given - read it, run it, move on.
# request, kinds that answer it (None: the shop does not sell it), colour rule, who it is for
TESTS = [
    ("I'm looking for a red t-shirt for men.", ["Tshirts"], "Red", "Men"),
    ("A brown leather wallet for my dad.", ["Wallets"], "Brown", "Men"),
    ("Comfortable sandals for summer, for a woman.", ["Sandals", "Flip Flops"], None, "Women"),
    ("A watch to wear at a business meeting.", ["Watches"], None, None),
    ("Running shoes for a beginner.", ["Sports Shoes"], None, None),
    ("A black handbag for work.", ["Handbags"], "Black", None),
    ("A white shirt for a job interview.", ["Shirts"], "White", None),
    ("Heels for a wedding.", ["Heels"], None, None),
    ("A t-shirt, any colour except black.", ["Tshirts"], "!Black", None),
    ("A kurta for a festival.", ["Kurtas"], None, None),
    ("A pink top for my girlfriend.", ["Tops"], "Pink", "Women"),
    ("Casual shoes, anything but white.", ["Casual Shoes"], "!White", None),
    ("Une montre pour femme.", ["Watches"], None, "Women"),
    ("Des tongs pour la plage.", ["Flip Flops"], None, None),
    ("Un sac à main rouge.", ["Handbags"], "Red", None),
    ("Une chemise bleue pour homme.", ["Shirts"], "Blue", "Men"),
    ("Des baskets de sport blanches.", ["Sports Shoes"], "White", None),
    ("Un portefeuille noir.", ["Wallets"], "Black", None),
    ("A warm winter coat.", None, None, None),
    ("Slim-fit jeans.", None, None, None),
    ("A dress for a party.", None, None, None),
    ("Sunglasses for the summer.", None, None, None),
    ("A tie to go with my suit.", None, None, None),
    ("Un manteau d'hiver.", None, None, None),
    ("Une écharpe en laine.", None, None, None),
    ("Un maillot de bain.", None, None, None),
]
FRENCH = ("Une ", "Un ", "Des ")


ALLOWED = {"Men": {"Men", "Unisex", "Boys"}, "Women": {"Women", "Unisex", "Girls"}}
KIDS = meta["title"].str.contains("Kid", case=False).to_numpy()   # the shop files most kids' items as "Unisex"


def is_right(row, test):
    """Is this product a right answer to this request?
    colour: "Red" (or "Maroon|Burgundy": any of them), "!Black" (or "!Pink|White": none of them)."""
    _, ok_kinds, colour, gender = test
    m = meta.iloc[row]
    if m.article_type not in ok_kinds:
        return False
    if colour and colour.startswith("!") and m.colour in colour[1:].split("|"):
        return False
    if colour and not colour.startswith("!") and m.colour not in colour.split("|") \
            and not (colour == "Blue" and m.colour == "Navy Blue"):
        return False
    if gender in ("Boys", "Girls"):
        return m.gender == gender or (m.gender == "Unisex" and KIDS[row])
    if gender and m.gender not in ALLOWED[gender]:
        return False
    return True


def grade(test, answer):
    rows = cited(answer)
    if test[1] is None:                       # we do not sell it: right if nothing is recommended
        return not rows
    return bool(rows) and all(is_right(r, test) for r in rows)


def evaluate(assistant, name="", tests=None, quiet=False):
    """Run an assistant on every request of the test set and grade each answer."""
    tests = tests or TESTS
    before, t0 = dict(USAGE), time.time()

    def one(test):
        out = assistant(test[0])
        return out[0] if isinstance(out, tuple) else out

    with ThreadPoolExecutor(4) as pool:
        answers_ = list(pool.map(one, tests))
    ok = [grade(t, a) for t, a in zip(tests, answers_)]
    sold = [i for i, t in enumerate(tests) if t[1] is not None]
    fr = [i for i in sold if tests[i][0].startswith(FRENCH)]
    en = [i for i in sold if i not in fr]
    no = [i for i, t in enumerate(tests) if t[1] is None]
    calls = USAGE["calls"] - before["calls"]
    score = {"name": name, "right": sum(ok[i] for i in sold), "of": len(sold),
             "english": sum(ok[i] for i in en), "of_english": len(en),
             "french": sum(ok[i] for i in fr), "of_french": len(fr),
             "said_no": sum(ok[i] for i in no), "of_no": len(no),
             "tokens_in": (USAGE["in"] - before["in"]) / len(tests),
             "tokens_out": (USAGE["out"] - before["out"]) / len(tests),
             "calls": calls / len(tests), "answers": answers_, "ok": ok}
    score["cents_per_answer"] = (score["tokens_in"] * 0.10 + score["tokens_out"] * 0.50) / 1e6 * 100
    if not quiet:
        for t, a, good in zip(tests, answers_, ok):
            shown = " ".join(a.split())[:90]
            print(f"{'✓' if good else '✗'}  {t[0]:<46s} cites {len(cited(a))}   {shown}")
    print(f"\n{name}  we sell it: right {score['right']}/{score['of']}"
          + (f"  (English {score['english']}/{score['of_english']}, French "
             f"{score['french']}/{score['of_french']})" if fr else "")
          + f"   we do not: said no {score['said_no']}/{score['of_no']}"
          f"   |  {score['calls']:.1f} calls, {score['tokens_in']:.0f} tokens in, "
          f"{score['tokens_out']:.0f} out per answer   {time.time() - t0:.0f} s")
    return score


v1 = evaluate(assistant_v1, "search, then answer")

### 2.4 ▶ · The trap, on purpose

One request from the test set, in detail: what the search found, and what the
assistant said.

In [ ]:
# Given - read it, run it, move on.
answer, rows = assistant_v1("Un sac à main rouge.")
show_products(rows, '"Un sac à main rouge." - what the search found')
print(answer)
print("\nred handbags in the catalogue:",
      int(((meta["article_type"] == "Handbags") & (meta["colour"] == "Red")).sum()))

### ❓ Checkpoint 2

1. Write down the score: right answers for the products we sell, in English and
   in French, and how many of the 8 requests for products we do not sell got a
   "no". Compare English with French.
2. In 2.4, is the answer faithful to the products the LLM was given? Is it
   true? Who made the mistake: the LLM or the search?
3. Last week's sentence expert learned from the shop's titles. In which
   language were they written, and what does that tell you about 2.4?

**⭐ Going further - optional**

4. Run `assistant_v1` on 2.4 with `k=30`. Does pasting more products save it?
   What does it cost?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 3 - The LLM writes the search

**~20 minutes.** One cell to write, three lines.

In checkpoint 2 the search heard the customer's exact words - French words,
and *"except black"*, which a search cannot hear. Now the LLM gets the search
as a **tool**: it decides what to search for, in English, and which **filters**
to set on the columns the shop filled in.

### 3.1 ✏️ · The tool

`search_catalogue` is what the LLM can call. Start from a mask that keeps every
product, then narrow it, one filter at a time:

- `colour`: keep the products whose colour contains it (`"blue"` keeps Navy Blue too);
- `exclude_colour`: drop the products whose colour contains it;
- `gender`: keep that gender and `"Unisex"`.

Colours are compared in lower case: the LLM writes `"red"` as often as `"Red"`.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 3 lines (plus their if)
#     if colour:         keep &= colours.str.contains(colour.lower()).to_numpy()
#     if exclude_colour: keep &= ~colours.str.contains(...).to_numpy()
#     if gender:         keep &= meta["gender"].isin([gender, "Unisex"]).to_numpy()
# ──────────────────────────────────────────────────────────────────────
def search_catalogue(query, colour=None, exclude_colour=None, gender=None, memory=None):
    """The tool: search the catalogue with filters. Returns the products as lines of text."""
    keep = np.ones(len(meta), dtype=bool)
    colours = meta["colour"].str.lower()
    ...          # <- replace this line
    return as_lines(search(query, keep=keep, memory=memory))


print(search_catalogue("t-shirt", exclude_colour="Black"))

### 3.2 ▶ · How the LLM is told about the tool, and the loop

The LLM never runs code. It receives a **description** of the tool (its name,
what it does, its arguments). When it wants to search, it stops and answers
with a request: *call `search_catalogue` with these arguments*. We run the
function, send back what it returned, and ask again - until it answers the
customer.

In [ ]:
# Given - read it, run it, move on.
TOOL = {
    "name": "search_catalogue",
    "description": "Search the shop's catalogue and get the closest products, one per line, "
                   "with their [id]. Write the query in English and describe the product itself "
                   "(kind, colour, style). Use the filters for what a search cannot hear: a "
                   "colour to keep or to exclude, who it is for.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "an English description of the product"},
            "colour": {"type": "string", "description": "keep only this colour, e.g. Red"},
            "exclude_colour": {"type": "string", "description": "drop this colour, e.g. Black"},
            "gender": {"type": "string", "enum": ["Men", "Women"]},
        },
        "required": ["query"],
    },
}

SYSTEM_TOOL = ("You are the shopping assistant of an online fashion shop. Use search_catalogue to "
               "find products, then answer from the products it returns only. Recommend at most "
               "3, and cite each one you recommend as [id]. If none fits, say so. Answer in the "
               "customer's language.")


def assistant(request, system=SYSTEM_TOOL, memory=None, verbose=False, tool=None, run_tool=None):
    """The LLM, with the search as a tool. Returns its answer to the customer.
    `tool` / `run_tool`: another tool description and the function that runs it (checkpoint 6)."""
    tool, run_tool = tool or TOOL, run_tool or search_catalogue
    messages = [{"role": "user", "content": request}]
    for _ in range(4):                                   # at most four rounds
        reply = llm(system=system, tools=[tool], messages=messages)
        if reply.stop_reason != "tool_use":              # it answered the customer: done
            return text_of(reply)
        messages.append({"role": "assistant", "content": reply.content})
        results = []
        for block in reply.content:
            if block.type == "tool_use":                 # it asked for a search: run it
                found = run_tool(**block.input, memory=memory)
                if verbose:
                    print("the LLM calls  search_catalogue(" + ", ".join(
                        f"{k}={v!r}" for k, v in block.input.items()) + ")")
                    show_products(cited(found), "what the tool sends back")
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": found or "no product"})
        messages.append({"role": "user", "content": results})
    return "(no answer after four rounds)"


for request in ("Un sac à main rouge.", "A t-shirt, any colour except black."):
    print(f"### {request}")
    answer = assistant(request, verbose=True)
    print(answer, "\n")

### 3.3 ▶ · The same 26 requests

In [ ]:
# Given - read it, run it, move on.
v2 = evaluate(assistant, "the LLM writes the search")

### ❓ Checkpoint 3

1. In 3.2, what query and which filters did the LLM write for *"Un sac à main
   rouge"*? For *"any colour except black"*?
2. Write down the new score next to checkpoint 2's, in English and in French.
   Which requests changed, and why?
3. Compare the tokens per answer with checkpoint 2. Where do the extra tokens
   come from? At Haiku's price ($0.10 per million tokens in, $0.50 out), what
   does one answer cost?

**⭐ Going further - optional**

4. Add a `usage` filter to the tool (`"Sports"`, `"Formal"`, `"Ethnic"`...):
   one property in `TOOL`, one line in `search_catalogue`. Which request of the
   test set could use it?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 4 - "We do not sell that"

**~10 minutes.** One cell to write: a sentence.

### 4.1 ▶ · Eight requests just next to what we sell

The 8 requests of the test set for products we do not sell (a coat, jeans...)
are far from everything in the shop. These eight are **neighbours**: a
backpack is almost a handbag, socks go with sports shoes, a belt with a
wallet. The search returns something plausible every time. Does the
assistant still say no?

In [ ]:
# Given - read it, run it, move on.
NEIGHBOURS = [(request, None, None, None) for request in (
    "A backpack for school.", "Running socks.", "A leather belt for men.", "Shorts for the gym.",
    "A sports bra.", "Un sac à dos pour la randonnée.", "Des chaussons pour la maison.",
    "A cap for the sun.")]

before = evaluate(assistant, "tool, no rule", tests=NEIGHBOURS)

### 4.2 ✏️ · Tell it what the shop sells

The LLM does not know that our shop sells twelve kinds of products and nothing
else. Write `RULE`: one or two sentences, added to the system text, that give
it the list (`kinds`) and say what to do with any other request. Then the same
eight neighbours, and the full test set to check that the rule costs nothing
on the products we do sell.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 1 sentence
#     a string that lists the kinds we sell (", ".join(kinds))
#     and says what to answer when the customer asks for anything else
# ──────────────────────────────────────────────────────────────────────
RULE = ...          # <- replace this line

after = evaluate(lambda request: assistant(request, SYSTEM_TOOL + " " + RULE), "tool + rule",
                 tests=NEIGHBOURS)
v3 = evaluate(lambda request: assistant(request, SYSTEM_TOOL + " " + RULE), "tool + rule",
              quiet=True)

### ❓ Checkpoint 4

1. How many of the 8 neighbours got a "no" before the rule, and after? Read
   one ✗ of 4.1: what did the assistant do instead of saying no?
2. On the full test set, did the rule cost anything on the 18 requests for
   products we sell? Compare the calls and the tokens per answer with
   checkpoint 3: why did they go down?
3. A customer asks for *"a bag for my laptop"*. We sell handbags. Should the
   assistant say no? Who decides - the LLM, or you?

*Your answers:*

1.
2.
3.

---

# Checkpoint 5 - The search is the weak link

**~15 minutes.** One cell to write, two lines.

The assistant has three parts: the LLM, the filters, and the memory - last
week's search. Keep the first two, and swap the memory.

### 5.1 ▶ · Four memories

- **photos, trained heads** - last week's search, the one you have used so far;
- **titles, sentence expert alone** - no heads: the request is compared with
  the 6,000 titles directly;
- **stale index** - next month, the heads are retrained (same recipe, another
  seed: the same quality). The requests go through the **new** text head, but
  nobody rebuilt the photo index: it still comes from the **old** photo head;
- **photos, untrained heads** - the two heads as they were before the game.

The first number is the memory alone, with no LLM and no filter: for the 18
requests we sell, does the search put a right product in its first 8?

In [ ]:
# Given - read it, run it, move on.
set_seed(SEED)
untrained = Heads().eval()
heads_next_month = train_heads(seed=1)


class TitleMemory:
    """No heads: the request is compared with the 6,000 titles directly."""

    def similarities(self, text):
        return title_codes @ sentence_code(text)


MEMORIES = {
    "photos, trained heads": MEMORY,
    "titles, sentence expert alone": TitleMemory(),
    "stale index (new heads, old photos)": PhotoMemory(heads_next_month, index_heads=heads),
    "photos, untrained heads": PhotoMemory(untrained),
}


def search_alone(memory):
    """For the 18 requests we sell: does the search alone put a right product in its top 8?"""
    sold = [t for t in TESTS if t[1] is not None]
    return sum(any(is_right(r, t) for r in search(t[0], memory=memory)) for t in sold), len(sold)


for name, memory in MEMORIES.items():
    hit, of = search_alone(memory)
    print(f"{name:<38s} the search alone finds a right product for {hit:>2}/{of} requests")

### 5.2 ✏️ · The whole assistant, on each memory

Run `evaluate` once per memory, with the assistant of checkpoint 4 (tool and
rule) and that memory. Keep the four scores in a dictionary. About a minute
and a half.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about 2 lines
#     for each (name, memory) in MEMORIES.items():
#        evaluate(lambda request, m=memory: assistant(request, SYSTEM_TOOL + " " + RULE, m), name, quiet=True)
# ──────────────────────────────────────────────────────────────────────
scores = ...          # <- replace this line

In [ ]:
# Given - read it, run it, move on.
names = list(MEMORIES)
alone = [search_alone(MEMORIES[n])[0] / 18 for n in names]
whole = [scores[n]["right"] / scores[n]["of"] for n in names]

fig, ax = plt.subplots(figsize=(8.6, 3.4))
x = np.arange(len(names))
b1 = ax.bar(x - 0.2, alone, 0.4, color=PALETTE["teal_dim"], label="the search alone, top 8")
b2 = ax.bar(x + 0.2, whole, 0.4, color=PALETTE["teal"], label="the whole assistant, right answers")
ax.bar_label(b1, fmt="{:.0%}", fontsize=8); ax.bar_label(b2, fmt="{:.0%}", fontsize=8)
ax.set_xticks(x, [textwrap.fill(n, 18) for n in names], fontsize=8)
ax.set_ylim(0, 1.15); ax.legend(fontsize=8, loc="upper right")
ax.set_title("Checkpoint 5 - same LLM, same filters, four memories (18 requests we sell)")
plt.show()

### ❓ Checkpoint 5

1. Fill the table: for each memory, the search alone and the whole assistant.
   Which part of the system decides the score?
2. The stale index: the new heads are as good as the old ones, and yet. Why
   does it fail? The assistant still gets some requests right - which part
   saves them, and why is that **dangerous** rather than reassuring?
3. You want to replace the photo memory by the title memory. Which number do
   you show the person who decides, and which number do you **not** show?

**⭐ Going further - optional**

4. Re-encoding the 6,000 titles takes the sentence expert a few seconds on this
   laptop (time it: `sentence_expert.encode(list(meta["title"]))`). How long
   for a shop with 10 million products? Who has to remember to do it?

*Your answers:*

1.
2.
3.
⭐ 4.

---

# Checkpoint 6 - The challenge

**Open.** Build your best assistant, and score it on **14 requests you have not
seen**: you tuned on the 26 of the test set, so they no longer measure
anything honestly. These fourteen are harder on purpose: two colours to avoid
at once, children's products, a colour the shop names differently.

### The levers

| Lever | What you change |
|---|---|
| the tool | what the LLM can ask for: a **list** of colours to exclude, children (`"Boys"`, `"Girls"`), `usage`... One property in the description, one line in the function |
| the system text | how strict it must be with the customer's constraints, what to do when nothing fits |
| the memory | `memory=` - photos, titles, or both (average the two `similarities`) |
| how many products | `k` in `search`, inside the tool's function |
| the thinking | `output_config={"effort": "medium"}` in `llm`: slower, dearer - better? |

To try another tool, write its description and its function, and pass both:
`assistant(request, system, tool=MY_TOOL, run_tool=my_search_catalogue)`.

### At 16:50

Out loud: **your score** on the 14 new requests, your **cost per answer**, and
**which lever moved it**.

In [ ]:
# Given - read it, run it, move on.
# Fourteen requests nobody tuned on. Same grade as the test set.
CHALLENGE = [
    ("A handbag, neither black nor brown.", ["Handbags"], "!Black|Brown", None),
    ("Sandals for my 8-year-old son.", ["Sandals"], None, "Boys"),
    ("A formal shirt for a man, not white.", ["Shirts"], "!White", "Men"),
    ("Something traditional for a man to wear at a wedding.", ["Kurtas"], None, "Men"),
    ("Des chaussures de ville marron pour homme.", ["Casual Shoes"], "Brown", "Men"),
    ("A gold watch for a woman.", ["Watches"], "Gold", "Women"),
    ("Running shoes for a woman, but not pink or white.", ["Sports Shoes"], "!Pink|White", "Women"),
    ("Un t-shirt bordeaux.", ["Tshirts"], "Maroon|Burgundy", None),
    ("Navy blue sports shoes for men.", ["Sports Shoes"], "Navy Blue", "Men"),
    ("A top for my daughter, she is ten.", ["Tops"], None, "Girls"),
    ("A beanie for the winter.", None, None, None),
    ("Une ceinture marron.", None, None, None),
    ("Earrings for a party.", None, None, None),
    ("Un parapluie.", None, None, None),
]

baseline = evaluate(lambda request: assistant(request, SYSTEM_TOOL + " " + RULE),
                    "checkpoint 4, unchanged", tests=CHALLENGE)

In [ ]:
# ──────────────────────────────────────────────────────────────────────
#  ✏️  YOUR TURN  -  about as many as you like
#     build my_assistant(request) -> answer, with the levers of the table
#     read the ✗ of the baseline first: each one tells you which lever to pull
# ──────────────────────────────────────────────────────────────────────
def my_assistant(request):
    return assistant(request, SYSTEM_TOOL + " " + RULE)      # <- start from here, change the levers


mine = evaluate(my_assistant, "my assistant", tests=CHALLENGE)

### ❓ Checkpoint 6

1. Your score on the 14 new requests, against checkpoint 4 unchanged. Your
   cost per answer, in cents.
2. Read the ✗ of the baseline. Which lever fixed which request? Which lever
   did nothing?
3. Your assistant scored on 26 requests you tuned on, then on 14 you did not.
   Which of the two numbers goes in your report, and why?

*Your answers:*

1.
2.
3.

---

## What to remember from today

- The LLM alone **speaks well and knows nothing of our shop**: asked for three
  of our products, it names none.
- Paste what the search found, and it answers **from the catalogue** - but it
  can only be as right as what it was given. A search that misses produces a
  polite, faithful and **false** answer.
- Let the LLM **write the search**: English queries, filters on the columns the
  shop already has. The weak requests of checkpoint 2 come back.
- Measure the **whole system** on a fixed test set, and keep requests you never
  tuned on for the final number.
- Swap the memory and the score follows: **a representation is only as good as
  what it does downstream, and that is what you measure.**

This was the last lab. Five sessions built representations; today one of them
went into a product and the product told you which of them was worth having.